In [2]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn.functional as F

# 1. Setup the project root directory paths safely
PROJECT_ROOT = r"C:\Users\shubh\projects\FairFraud"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import your standardized data pipeline
from src.pipeline import get_data

print("🔄 Fetching pipeline data splits for new inference data...")
data = get_data()

# 🔍 Simulating "New Data" using your test set matrices
X_raw = data["X_test_raw"]         # Used by CatBoost
X_sparse = data["X_test_sparse"]   # Used by LightGBM and XGBoost
X_dense = data["X_test_dense"]     # Used by TabNet, FT-Transformer, and SAINT

MODELS_DIR = Path(PROJECT_ROOT) / "models"
all_probabilities = {}

# ==============================================================================
# LOAD AND PREDICT: ALL 7 MODELS
# ==============================================================================

# 1. CatBoost (.cbm)
from catboost import CatBoostClassifier, Pool
print("🔮 Processing Model 1: CatBoost...")
cb = CatBoostClassifier()
cb.load_model(str(MODELS_DIR / "catboost_model.cbm"))
cb_pool = Pool(X_raw, cat_features=data["categorical_features"])
all_probabilities["CatBoost"] = cb.predict_proba(cb_pool)[:, 1]

# 2. LightGBM (.pkl)
import joblib
print("🔮 Processing Model 2: LightGBM...")
lgb_model = joblib.load(MODELS_DIR / "lightgbm_classifier.pkl")
all_probabilities["LightGBM"] = lgb_model.predict_proba(X_sparse)[:, 1]

# 3. XGBoost (.json)
import xgboost as xgb
print("🔮 Processing Model 3: XGBoost...")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(MODELS_DIR / "xgboost_model.json"))
all_probabilities["XGBoost"] = xgb_model.predict_proba(X_sparse)[:, 1]

# 4. TabNet (.zip)
from pytorch_tabnet.tab_model import TabNetClassifier
print("🔮 Processing Model 4: TabNet...")
tabnet = TabNetClassifier()
tabnet.load_model(str(MODELS_DIR / "tabnet_model.zip"))
all_probabilities["TabNet"] = tabnet.predict_proba(X_dense)[:, 1]

# 5. FT-Transformer (.pkl via Skorch)
from skorch import NeuralNetClassifier
print("🔮 Processing Model 5: FT-Transformer...")
# We initialize the model structure with skorch to easily load params
from predict import FastFTTransformerModule # Or import your module definition from its notebook
ft_model = joblib.load(MODELS_DIR / "ft_transformer_model.pkl") 
# If saved via skorch save_params instead: ft_model.load_params(f_params=...)
all_probabilities["FT-Transformer"] = ft_model.predict_proba(X_dense.astype(np.float32))[:, 1]

# 6. SAINT (.pkl via Skorch)
print("🔮 Processing Model 6: SAINT...")
saint_model = joblib.load(MODELS_DIR / "saint_model.pkl")
all_probabilities["SAINT"] = saint_model.predict_proba(X_dense.astype(np.float32))[:, 1]

# 7. GraphSAGE (.pt via PyTorch Geometric)
print("🔮 Processing Model 7: GraphSAGE...")
from predict import GraphSAGEModel # Make sure your GraphSAGE class structure is loaded in current memory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gs_model = GraphSAGEModel(in_features=X_dense.shape[1], hidden_dim=32).to(device)
gs_model.load_state_dict(torch.load(MODELS_DIR / "graphsage_model.pt", map_location=device))
gs_model.eval()

# GraphSAGE requires an edge index. Building localized temporal evaluation index:
num_nodes = X_dense.shape[0]
k_neighbors = 3
src_nodes = np.repeat(np.arange(num_nodes), k_neighbors)
tgt_nodes = np.array([np.random.randint(max(0, i-50), min(num_nodes, i+50), k_neighbors) for i in range(num_nodes)]).flatten()
eval_edge_index = torch.tensor(np.vstack([src_nodes, tgt_nodes]), dtype=torch.long).to(device)

with torch.no_grad():
    gs_tensor = torch.tensor(X_dense, dtype=torch.float32).to(device)
    gs_outputs = F.softmax(gs_model(gs_tensor, eval_edge_index), dim=1)
    all_probabilities["GraphSAGE"] = gs_outputs[:, 1].cpu().numpy()

# ==============================================================================
# ENSEMBLE COMBINATION & DECISION MAKING
# ==============================================================================
print("\n🤝 Aggregating all 7 model scores into a final prediction...")

# 1. Compute the average probability across all 7 models
final_probabilities = np.mean(list(all_probabilities.values()), axis=0)

# 2. Apply your operational decision threshold (e.g., 0.50 or your custom fixed FPR threshold)
# Set your threshold lower if you want to capture more fraud, or leave at 0.50 for baseline
DECISION_THRESHOLD = 0.50 
final_predictions = (final_probabilities >= DECISION_THRESHOLD).astype(int)

# 3. Create a clean results DataFrame
results_df = pd.DataFrame({
    'Fraud_Probability': final_probabilities,
    'Prediction_Class': final_predictions
})

# Map numbers to readable words
results_df['Prediction_Label'] = results_df['Prediction_Class'].map({0: 'Legitimate', 1: 'Fraud'})

print("✅ Predictions successfully completed!")
print(results_df['Prediction_Label'].value_counts())


🔄 Fetching pipeline data splits for new inference data...
🔮 Processing Model 1: CatBoost...
🔮 Processing Model 2: LightGBM...


AttributeError: 'Booster' object has no attribute 'predict_proba'